In [0]:
from pyspark.sql import functions as F
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")
T_email = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")

In [0]:
start_date = "2026-05-01"
end_date   = "2026-05-30"

snapshot_date_id = "20260530"

In [0]:
df_set_to_renew = (
    T_subscriber.filter(F.col("snapshot_date_id") == '20260430')
    .filter((F.to_date(T_subscriber.renewal_datetime).between(start_date, end_date)))
    .filter((F.col("plan_id") == "annual1"))
    .select('cde_id')
    .distinct()
)
df_set_to_renew.count()

In [0]:
df_renewed = (
    df_set_to_renew
    .join(T_subscriber.filter(F.col("snapshot_date_id") == snapshot_date_id), on="cde_id", how="inner")
    .filter(F.col("renewal_status").like("%Active%"))
    .filter(F.col("plan_id") == "annual1")
    .select('cde_id')
)
df_renewed.count()

In [0]:
#Members that Joined on the Activated_at_datetime
df_active_annual_snapshot = (
    T_subscriber.filter(F.col("snapshot_date_id") == snapshot_date_id)
    .filter(F.to_date(T_subscriber.activated_at_datetime).between(start_date, end_date))
    .filter((F.col("plan_id") == "annual1") & (F.col("renewal_status").like("%Active%")))
    .select('cde_id',T_subscriber.activated_at_datetime)
    .distinct()
)

#All instances of Activation
df_first_activation_new = (
    T_subscriber
    .groupBy("cde_id")
    .agg(
        F.min(F.to_date(T_subscriber.activated_at_datetime))
    )
    .withColumnRenamed("min(to_date(activated_at_datetime))","first_activation")
)

#Joining the 2 tables
df_details = (df_active_annual_snapshot
              .join(df_first_activation_new, on="cde_id", how="inner")
              )

#New Members
df_new_signups = (
    df_details
    .filter(F.col("first_activation").between(start_date, end_date))
    .select('cde_id')
)
df_returning_signups = (
    df_details
    .filter(~F.col("first_activation").between(start_date, end_date))
    .select('cde_id')
)

df_new_signups.count()

In [0]:
df_returning_labeled = (
    df_returning_signups
    .join(
        df_renewed.select("cde_id").distinct().withColumn("is_renewal", F.lit(1)),
        on="cde_id", how="left"
    )
    .join(
        df_new_signups.select("cde_id").distinct().withColumn("is_new", F.lit(1)),
        on="cde_id", how="left"
    )
    .fillna(0, subset=["is_renewal", "is_new"])
)

In [0]:
df_scheduled_upgrades_LM = (
    T_subscriber.filter(F.col("snapshot_date_id").between('20260401', '20260430'))
    .filter((F.col("renewal_status") == "Active_Annual_Upgrade"))
    .filter(F.to_date(F.col("renewal_datetime")).between(start_date, end_date))
    .select('cde_id')
    .distinct()
)

df_scheduled_upgrades_LM.count() 

In [0]:
df_upgraded_LM = (
    df_scheduled_upgrades_LM
    .join(T_subscriber.filter(F.col("snapshot_date_id") == snapshot_date_id), on="cde_id", how="inner")
    .filter(F.col("renewal_status").like("%Active%"))
    .filter(F.col("plan_id") == "annual1")
    .select('cde_id')
)

df_upgraded_LM.count()

In [0]:
df_scheduled_upgrades_TM = (
    T_subscriber.filter(F.col("snapshot_date_id").between('20260501', '20260530'))
    .filter((F.col("renewal_status") == "Active_Annual_Upgrade"))
    .select('cde_id')
    .distinct()
    .join(df_scheduled_upgrades_LM, on="cde_id", how="left_anti")
)

df_scheduled_upgrades_TM.count() 

In [0]:
df_scheduled_upgrades_TM.display()

In [0]:
df_upgraded_TM = (
    df_scheduled_upgrades_TM
    .join(T_subscriber.filter(F.col("snapshot_date_id") == snapshot_date_id), on="cde_id", how="inner")
    .filter(F.col("renewal_status").like("%Active%"))
    .filter(F.col("plan_id") == "annual1")
    .select('cde_id')
)

df_upgraded_TM.count()

In [0]:
# 1. Renewals — base layer, labeled
df_labeled_renewals = (
    df_renewed.select("cde_id").distinct()
    .withColumn("member_type", F.lit("Renewal"))
)

# 2. Net New — append, exclude any cde_id already in renewals
df_labeled_new = (
    df_new_signups.select("cde_id").distinct()
    .join(df_labeled_renewals.select("cde_id"), on="cde_id", how="left_anti")
    .withColumn("member_type", F.lit("Net New"))
)

# 3. Upgrades from last month — append, exclude any cde_id already labeled
df_labeled_so_far = df_labeled_renewals.unionByName(df_labeled_new)

df_labeled_upgrades_LM = (
    df_upgraded_LM.select("cde_id").distinct()
    .join(df_labeled_so_far.select("cde_id"), on="cde_id", how="left_anti")
    .withColumn("member_type", F.lit("Upgrades (LM)"))
)

# 4. Upgrades from this month — append, exclude any cde_id already labeled
df_labeled_so_far = df_labeled_so_far.unionByName(df_labeled_upgrades_LM)

df_labeled_upgrades_TM = (
    df_upgraded_TM.select("cde_id").distinct()
    .join(df_labeled_so_far.select("cde_id"), on="cde_id", how="left_anti")
    .withColumn("member_type", F.lit("Upgrades (TM)"))
)

# 5. Returning — append, exclude any cde_id already labeled above
df_labeled_so_far = df_labeled_so_far.unionByName(df_labeled_upgrades_TM)

df_labeled_returning = (
    df_returning_signups.select("cde_id").distinct()
    .join(df_labeled_so_far.select("cde_id"), on="cde_id", how="left_anti")
    .withColumn("member_type", F.lit("Returning").cast("string"))
)

# Final union
df_member_segments = (
    df_labeled_renewals
    .unionByName(df_labeled_new)
    .unionByName(df_labeled_upgrades_LM)
    .unionByName(df_labeled_upgrades_TM)
    .unionByName(df_labeled_returning)
)

df_member_segments.groupBy("member_type").count().display()
df_member_segments.count()

In [0]:
df_unlabeled = df_member_segments.filter(F.col("member_type") == "Upgrades (TM)").select("cde_id")

df_pre_may_state = (
    T_subscriber.filter(F.col("snapshot_date_id") == '20260430')
    .select(
        "cde_id",
        F.col("renewal_status").alias("pre_renewal_status"),
        F.col("plan_id").alias("pre_plan_id"),
    )
)

df_current_state = (
    T_subscriber.filter(F.col("snapshot_date_id") == snapshot_date_id)
    .select(
        "cde_id",
        F.col("renewal_status").alias("current_renewal_status"),
        F.col("plan_id").alias("current_plan_id"),
    )
)

df_unlabeled_diagnostic = (
    df_unlabeled
    .join(df_pre_may_state, on="cde_id", how="left")
    .join(df_current_state, on="cde_id", how="left")
    .groupBy("pre_renewal_status", "pre_plan_id", "current_renewal_status", "current_plan_id")
    .count()
    .orderBy(F.desc("count"))
)
df_unlabeled_diagnostic.display()

In [0]:
from pyspark.sql import Window

# Normalize first — uppercase + trim — then dedupe within cde_id
T_email_normalized = (
    T_email
    .withColumn("Email_Address", F.upper(F.trim(F.col("Email_Address"))))
    .dropDuplicates(["cde_id", "Email_Address"])
)

w = Window.partitionBy("cde_id").orderBy("Email_Address")

df_with_rank = T_email_normalized.withColumn("rn", F.row_number().over(w))

T_email_deduped = (
    df_with_rank
    .groupBy("cde_id")
    .agg(
        F.first(F.when(F.col("rn") == 1, F.col("Email_Address"))).alias("Primary_Email"),
        F.first(F.when(F.col("rn") == 2, F.col("Email_Address")), ignorenulls=True).alias("alternate_email"),
    )
)

In [0]:
final_send = (
    df_member_segments
)


In [0]:
final_send.display()